In [1]:
# Install all required packages
!pip install sentence-transformers
!pip install langdetect
!pip install deep-translator
!pip install ollama
!pip install numpy
!pip install faiss-cpu
!pip install pickle5

ERROR: Could not find a version that satisfies the requirement pickle5 (from versions: none)
ERROR: No matching distribution found for pickle5


In [2]:
import sqlite3
import numpy as np
import os
import json
import pickle
import warnings
import faiss
warnings.filterwarnings("ignore")

from sentence_transformers import SentenceTransformer
from langdetect import detect, DetectorFactory
from deep_translator import GoogleTranslator
import ollama

# Fix random seed for consistent language detection
DetectorFactory.seed = 0

print("✅ All imports successful")
print(f"✅ FAISS version: {faiss.__version__}")

✅ All imports successful
✅ FAISS version: 1.14.3


In [3]:
def create_database():
    """
    Create SQLite database with English content.
    All content stored in English only.
    FAISS handles the vector search separately.
    """
    
    conn = sqlite3.connect("knowledge_base.db")
    cursor = conn.cursor()
    
    # Create table
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS knowledge (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            category TEXT NOT NULL,
            title TEXT NOT NULL,
            content TEXT NOT NULL
        )
    """)
    
    # Sample English data
    sample_data = [
        (
            "health",
            "Dengue Fever Symptoms",
            "Dengue fever symptoms include high fever, severe headache, pain behind the eyes, joint and muscle pain, rash, and mild bleeding. Symptoms usually appear 4 to 10 days after infection and last for 2 to 7 days."
        ),
        (
            "health",
            "Dengue Prevention",
            "To prevent dengue fever, eliminate standing water around your home, use mosquito repellent, wear long-sleeved clothing, use mosquito nets, and keep windows and doors closed or screened."
        ),
        (
            "health",
            "Malaria Treatment",
            "Malaria treatment depends on the type of malaria parasite and severity. Common treatments include antimalarial medications such as chloroquine, artemisinin-based combination therapies, and supportive care."
        ),
        (
            "education",
            "School Admission Process",
            "School admission requires birth certificate, vaccination records, previous school records if applicable, and parent identification documents. Applications must be submitted before the deadline."
        ),
        (
            "education",
            "Scholarship Eligibility",
            "Scholarships are available for students with academic excellence, financial need, or special talents. Students must maintain a minimum GPA of 3.5 and submit applications by March 31 each year."
        ),
        (
            "government",
            "National ID Card Application",
            "To apply for a national ID card, you need to visit the nearest divisional secretariat with your birth certificate, Grama Niladhari certificate, and two passport-size photographs."
        ),
        (
            "government",
            "Passport Application Process",
            "Passport application requires national ID card, birth certificate, police clearance report, and completed application form. Processing takes 3 to 10 working days for normal service."
        ),
        (
            "agriculture",
            "Paddy Cultivation Season",
            "Sri Lanka has two main paddy cultivation seasons. Maha season runs from October to March and Yala season runs from April to September. Farmers receive government subsidies for seeds and fertilizer."
        ),
        (
            "agriculture",
            "Fertilizer Subsidy Program",
            "The government provides fertilizer subsidies to registered farmers. Farmers must register at the nearest agricultural office with land deed or lease agreement to receive subsidized fertilizer."
        ),
        (
            "finance",
            "Bank Loan Application",
            "To apply for a bank loan, you need income proof, bank statements for last 6 months, national ID, and collateral documents if applicable. Loan approval typically takes 5 to 7 working days."
        ),
        (
            "finance",
            "Samurdhi Benefits",
            "Samurdhi is a government welfare program providing financial assistance to low-income families. Eligible families receive monthly allowances and can access low-interest loans through Samurdhi banks."
        ),
        (
            "health",
            "COVID-19 Vaccination",
            "COVID-19 vaccines are available at government hospitals and health centers. Bring your national ID card for registration. Booster doses are recommended every 6 months for high-risk groups."
        ),
    ]
    
    # Clear old data and insert fresh
    cursor.execute("DELETE FROM knowledge")
    cursor.executemany(
        "INSERT INTO knowledge (category, title, content) VALUES (?, ?, ?)",
        sample_data
    )
    
    conn.commit()
    conn.close()
    
    print(f"✅ Database created with {len(sample_data)} records")
    print("📂 File: knowledge_base.db")


create_database()

✅ Database created with 12 records
📂 File: knowledge_base.db


In [4]:
# Fix XetHub timeout issue
os.environ["HF_HUB_DISABLE_XET"] = "1"

print("⏳ Loading multilingual embedding model...")

embedding_model = SentenceTransformer("intfloat/multilingual-e5-small")

# Get embedding dimension (needed for FAISS index creation)
EMBEDDING_DIM = embedding_model.get_sentence_embedding_dimension()

print(f"✅ Embedding model loaded!")
print(f"📐 Embedding dimension: {EMBEDDING_DIM}")

⏳ Loading multilingual embedding model...


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

✅ Embedding model loaded!
📐 Embedding dimension: 384


In [5]:
def load_documents_from_db():
    """
    Load all documents from SQLite database.
    
    Important:
    - FAISS stores vectors by integer index (0, 1, 2...)
    - We maintain a separate doc_store list
    - FAISS index position maps to doc_store position
    
    Returns:
        List of document dictionaries
    """
    
    conn = sqlite3.connect("knowledge_base.db")
    cursor = conn.cursor()
    
    cursor.execute("SELECT id, category, title, content FROM knowledge")
    rows = cursor.fetchall()
    conn.close()
    
    documents = []
    for row in rows:
        documents.append({
            "db_id": row[0],           # Original SQLite row ID
            "category": row[1],
            "title": row[2],
            "content": row[3],
            # Combined text with E5 passage prefix
            "full_text": f"passage: {row[2]}. {row[3]}"
        })
    
    print(f"✅ Loaded {len(documents)} documents from SQLite")
    return documents


# Load documents
documents = load_documents_from_db()

✅ Loaded 12 documents from SQLite


In [6]:
# File paths for saving FAISS index
FAISS_INDEX_PATH = "faiss_index.bin"
DOC_STORE_PATH = "doc_store.pkl"


def build_faiss_index(documents):
    """
    Build FAISS index from document embeddings.
    
    FAISS Index Types Used:
    - IndexFlatIP → Inner Product (cosine similarity with normalized vectors)
    - Best for small-medium datasets (< 100k docs)
    - Exact search (no approximation)
    
    Args:
        documents: List of document dicts
    
    Returns:
        faiss_index: FAISS index object
        doc_store: List of documents (maps to FAISS positions)
    """
    
    print("⏳ Generating embeddings...")
    
    # Extract text for embedding
    texts = [doc["full_text"] for doc in documents]
    
    # Generate normalized embeddings
    # normalize=True → allows using Inner Product as cosine similarity
    embeddings = embedding_model.encode(
        texts,
        normalize_embeddings=True,
        show_progress_bar=True,
        batch_size=32,
        convert_to_numpy=True
    )
    
    # Ensure float32 (FAISS requirement)
    embeddings = embeddings.astype(np.float32)
    
    print(f"\n✅ Embeddings generated: {embeddings.shape}")
    
    # ============================================
    # Create FAISS Index
    # ============================================
    # IndexFlatIP = Flat Inner Product index
    # For normalized vectors → Inner Product = Cosine Similarity
    # Range: -1 to 1 (higher = more similar)
    
    faiss_index = faiss.IndexFlatIP(EMBEDDING_DIM)
    
    # Add embeddings to FAISS index
    # FAISS assigns integer IDs: 0, 1, 2, ... automatically
    faiss_index.add(embeddings)
    
    print(f"✅ FAISS index built!")
    print(f"📊 Total vectors in index: {faiss_index.ntotal}")
    print(f"📐 Index dimension: {faiss_index.d}")
    
    # doc_store maps FAISS position → document
    # FAISS position 0 → documents[0]
    # FAISS position 1 → documents[1]
    doc_store = documents
    
    return faiss_index, doc_store


def save_faiss_index(faiss_index, doc_store):
    """
    Save FAISS index and document store to disk.
    Avoids rebuilding index on every restart.
    """
    
    # Save FAISS binary index
    faiss.write_index(faiss_index, FAISS_INDEX_PATH)
    
    # Save document store (metadata)
    with open(DOC_STORE_PATH, "wb") as f:
        pickle.dump(doc_store, f)
    
    print(f"✅ FAISS index saved → {FAISS_INDEX_PATH}")
    print(f"✅ Document store saved → {DOC_STORE_PATH}")


def load_faiss_index():
    """
    Load existing FAISS index from disk.
    Returns None if not found.
    """
    
    if os.path.exists(FAISS_INDEX_PATH) and os.path.exists(DOC_STORE_PATH):
        
        faiss_index = faiss.read_index(FAISS_INDEX_PATH)
        
        with open(DOC_STORE_PATH, "rb") as f:
            doc_store = pickle.load(f)
        
        print(f"✅ FAISS index loaded from disk!")
        print(f"📊 Vectors in index: {faiss_index.ntotal}")
        
        return faiss_index, doc_store
    
    else:
        print("⚠️ No existing FAISS index found. Will build new one.")
        return None, None


# ============================================
# Load or Build FAISS Index
# ============================================
print("🔍 Checking for existing FAISS index...")

faiss_index, doc_store = load_faiss_index()

if faiss_index is None:
    print("🔨 Building new FAISS index...")
    faiss_index, doc_store = build_faiss_index(documents)
    save_faiss_index(faiss_index, doc_store)

print(f"\n✅ FAISS Ready!")
print(f"   Vectors: {faiss_index.ntotal}")
print(f"   Dimension: {faiss_index.d}")

🔍 Checking for existing FAISS index...
⚠️ No existing FAISS index found. Will build new one.
🔨 Building new FAISS index...
⏳ Generating embeddings...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


✅ Embeddings generated: (12, 384)
✅ FAISS index built!
📊 Total vectors in index: 12
📐 Index dimension: 384
✅ FAISS index saved → faiss_index.bin
✅ Document store saved → doc_store.pkl

✅ FAISS Ready!
   Vectors: 12
   Dimension: 384


In [7]:
# ============================================
# LANGUAGE CODE MAPPING
# ============================================
LANGUAGE_MAP = {
    "en": "English",
    "ta": "Tamil",
    "si": "Sinhala"
}


# ============================================
# UNICODE RANGES FOR SCRIPT DETECTION
# ============================================

# Sinhala Unicode Block: U+0D80 to U+0DFF
SINHALA_UNICODE_START = 0x0D80
SINHALA_UNICODE_END   = 0x0DFF

# Tamil Unicode Block: U+0B80 to U+0BFF
TAMIL_UNICODE_START   = 0x0B80
TAMIL_UNICODE_END     = 0x0BFF


def count_script_characters(text, start, end):
    """
    Count how many characters in text fall within
    a specific Unicode range.
    
    Args:
        text  : Input string
        start : Unicode range start (int)
        end   : Unicode range end (int)
    
    Returns:
        count : Number of characters in that range
    """
    return sum(1 for ch in text if start <= ord(ch) <= end)


def detect_language_unicode(text):
    """
    Detect language using Unicode character ranges.
    
    Detection Logic:
    ┌──────────────────────────────────────────────┐
    │ 1. Count Sinhala Unicode chars in text       │
    │ 2. Count Tamil Unicode chars in text         │
    │ 3. If Sinhala chars found → 'si'             │
    │ 4. If Tamil chars found   → 'ta'             │
    │ 5. If neither             → 'en' (default)   │
    └──────────────────────────────────────────────┘
    
    Why Unicode detection?
    - langdetect fails on short Sinhala text
    - Unicode ranges are script-specific and unique
    - Sinhala: U+0D80–U+0DFF (only used for Sinhala)
    - Tamil  : U+0B80–U+0BFF (only used for Tamil)
    - 100% accurate for these scripts
    
    Args:
        text : Input query string
    
    Returns:
        lang_code : 'si', 'ta', or 'en'
    """
    
    # Remove whitespace for accurate counting
    clean_text = text.strip()
    
    if not clean_text:
        return "en"
    
    # Count script-specific characters
    sinhala_count = count_script_characters(
        clean_text,
        SINHALA_UNICODE_START,
        SINHALA_UNICODE_END
    )
    
    tamil_count = count_script_characters(
        clean_text,
        TAMIL_UNICODE_START,
        TAMIL_UNICODE_END
    )
    
    total_chars = len(clean_text.replace(" ", ""))
    
    # Calculate script ratio
    sinhala_ratio = sinhala_count / total_chars if total_chars > 0 else 0
    tamil_ratio   = tamil_count   / total_chars if total_chars > 0 else 0
    
    print(f"   🔢 Unicode Analysis:")
    print(f"      Total chars   : {total_chars}")
    print(f"      Sinhala chars : {sinhala_count} ({sinhala_ratio:.1%})")
    print(f"      Tamil chars   : {tamil_count}   ({tamil_ratio:.1%})")
    
    # Decision (threshold: at least 10% script chars)
    SCRIPT_THRESHOLD = 0.10
    
    if sinhala_ratio >= SCRIPT_THRESHOLD:
        return "si"
    
    elif tamil_ratio >= SCRIPT_THRESHOLD:
        return "ta"
    
    else:
        return "en"


def detect_language_langdetect(text):
    """
    Fallback detection using langdetect library.
    Only used for English (since Unicode covers TA/SI).
    
    Args:
        text : Input query string
    
    Returns:
        lang_code : detected language or 'en'
    """
    try:
        detected = detect(text)
        return detected
    except Exception:
        return "en"


def detect_language(text):
    """
    Hybrid Language Detection:
    
    Strategy:
    ┌─────────────────────────────────────────────────┐
    │  Step 1: Unicode scan for Sinhala/Tamil         │
    │     → If found → return immediately (100% sure) │
    │                                                  │
    │  Step 2: If no Sinhala/Tamil Unicode found      │
    │     → Use langdetect for English confirmation   │
    │     → Default to 'en' if langdetect fails       │
    └─────────────────────────────────────────────────┘
    
    Why hybrid?
    - Unicode = perfect for Sinhala & Tamil
    - langdetect = handles English edge cases
    - Combined = best of both approaches
    
    Args:
        text : Input query (any language)
    
    Returns:
        lang_code : 'en', 'ta', or 'si'
    """
    
    print(f"\n🔍 Detecting language for: '{text[:50]}{'...' if len(text)>50 else ''}'")
    
    # ── Priority 1: Unicode-based detection ───────────────────
    # This is always checked first and trusted completely
    unicode_lang = detect_language_unicode(text)
    
    if unicode_lang in ["si", "ta"]:
        lang_name = LANGUAGE_MAP[unicode_lang]
        print(f"   ✅ Unicode Detection → {lang_name} ({unicode_lang})")
        return unicode_lang
    
    # ── Priority 2: langdetect for English ────────────────────
    # Only reached if no Sinhala/Tamil Unicode found
    print(f"   ℹ️  No Sinhala/Tamil Unicode found → checking langdetect")
    
    fallback_lang = detect_language_langdetect(text)
    
    # Only trust langdetect for English
    # If it returns something unexpected → default to English
    if fallback_lang == "en":
        print(f"   ✅ langdetect → English (en)")
        return "en"
    
    elif fallback_lang in ["ta", "si"]:
        # langdetect agrees with a known language
        lang_name = LANGUAGE_MAP.get(fallback_lang, fallback_lang)
        print(f"   ✅ langdetect → {lang_name} ({fallback_lang})")
        return fallback_lang
    
    else:
        # Unknown language → default to English
        print(f"   ⚠️  langdetect returned '{fallback_lang}' → defaulting to English")
        return "en"


# ============================================
# TRANSLATION FUNCTIONS (unchanged)
# ============================================

def translate_to_english(text, source_lang):
    """
    Translate Tamil or Sinhala query to English.
    Uses Google Translator (free, no API key needed).
    Returns original text if already English.
    """
    if source_lang == "en":
        print("   ✅ Query already in English, no translation needed.")
        return text
    
    try:
        translator = GoogleTranslator(source=source_lang, target="en")
        translated = translator.translate(text)
        print(f"   🔄 Translated to English: {translated}")
        return translated
    
    except Exception as e:
        print(f"   ⚠️ Translation failed: {e}. Using original text.")
        return text


def translate_response(text, target_lang):
    """
    Translate English LLM response back to user's language.
    Handles long responses by chunking (Google limit: 5000 chars).
    Returns original English if target is English.
    """
    if target_lang == "en":
        return text
    
    try:
        translator = GoogleTranslator(source="en", target=target_lang)
        
        if len(text) <= 4500:
            return translator.translate(text)
        
        # Chunk for long responses
        chunks = [text[i:i+4500] for i in range(0, len(text), 4500)]
        translated_chunks = [translator.translate(chunk) for chunk in chunks]
        return " ".join(translated_chunks)
    
    except Exception as e:
        print(f"   ⚠️ Response translation failed: {e}. Returning English.")
        return text


# ============================================
# TEST: Unicode Language Detection
# ============================================

print("🧪 Testing Unicode-based Language Detection")
print("=" * 55)

test_cases = [
    # (expected_lang, query)
    ("en", "What are the symptoms of dengue fever?"),
    ("en", "How to apply for a passport?"),
    ("ta", "டெங்கு காய்ச்சலின் அறிகுறிகள் என்ன?"),
    ("ta", "வங்கி கடன் விண்ணப்பிக்க என்ன தேவை?"),
    ("si", "ඩෙංගු උණේ රෝග ලක්ෂණ මොනවාද?"),
    ("si", "ජාතික හැඳුනුම්පත ඉල්ලුම් කරන්නේ කෙසේද?"),
    ("si", "පොහොර සහනාධාරය ලබා ගන්නේ කෙසේද?"),
    ("ta", "கல்வி உதவித்தொகைக்கு எப்படி விண்ணப்பிப்பது?"),
]

all_passed = True

for expected, query in test_cases:
    detected = detect_language(query)
    status = "✅ PASS" if detected == expected else "❌ FAIL"
    
    if detected != expected:
        all_passed = False
    
    print(f"\n{status}")
    print(f"  Query    : {query}")
    print(f"  Expected : {LANGUAGE_MAP[expected]} ({expected})")
    print(f"  Detected : {LANGUAGE_MAP.get(detected, detected)} ({detected})")

print("\n" + "=" * 55)
if all_passed:
    print("🎉 All language detection tests PASSED!")
else:
    print("⚠️  Some tests failed. Check Unicode ranges.")
print("=" * 55)

🧪 Testing Unicode-based Language Detection

🔍 Detecting language for: 'What are the symptoms of dengue fever?'
   🔢 Unicode Analysis:
      Total chars   : 32
      Sinhala chars : 0 (0.0%)
      Tamil chars   : 0   (0.0%)
   ℹ️  No Sinhala/Tamil Unicode found → checking langdetect
   ✅ langdetect → English (en)

✅ PASS
  Query    : What are the symptoms of dengue fever?
  Expected : English (en)
  Detected : English (en)

🔍 Detecting language for: 'How to apply for a passport?'
   🔢 Unicode Analysis:
      Total chars   : 23
      Sinhala chars : 0 (0.0%)
      Tamil chars   : 0   (0.0%)
   ℹ️  No Sinhala/Tamil Unicode found → checking langdetect
   ✅ langdetect → English (en)

✅ PASS
  Query    : How to apply for a passport?
  Expected : English (en)
  Detected : English (en)

🔍 Detecting language for: 'டெங்கு காய்ச்சலின் அறிகுறிகள் என்ன?'
   🔢 Unicode Analysis:
      Total chars   : 32
      Sinhala chars : 0 (0.0%)
      Tamil chars   : 31   (96.9%)
   ✅ Unicode Detection → Tamil (

In [8]:
def retrieve_with_faiss(query_english, top_k=3, score_threshold=0.3):
    """
    Retrieve most relevant documents using FAISS.
    
    How FAISS search works:
    1. Embed query as float32 vector
    2. FAISS searches index for nearest neighbors
    3. Returns distances (scores) and indices
    4. We map indices back to documents via doc_store
    
    Args:
        query_english: Query in English
        top_k: Number of top documents to retrieve
        score_threshold: Min cosine similarity score (0 to 1)
    
    Returns:
        List of matched documents with scores
    """
    
    # Step 1: Embed query with E5 prefix
    query_embedding = embedding_model.encode(
        [f"query: {query_english}"],
        normalize_embeddings=True,
        convert_to_numpy=True
    )
    
    # FAISS requires float32
    query_embedding = query_embedding.astype(np.float32)
    
    # Step 2: Search FAISS index
    # Returns:
    #   distances → similarity scores (shape: [1, top_k])
    #   indices   → FAISS positions   (shape: [1, top_k])
    distances, indices = faiss_index.search(query_embedding, top_k)
    
    # Step 3: Map results back to documents
    results = []
    
    for rank, (score, idx) in enumerate(zip(distances[0], indices[0])):
        
        # FAISS returns -1 for empty slots (if fewer docs than top_k)
        if idx == -1:
            continue
        
        # Apply score threshold filter
        if score < score_threshold:
            continue
        
        results.append({
            "document": doc_store[idx],     # Map FAISS index → document
            "score": float(score),          # Cosine similarity score
            "faiss_id": int(idx),           # FAISS internal ID
            "rank": rank + 1
        })
    
    # Display results
    print(f"\n📚 FAISS Retrieved {len(results)} Documents (threshold={score_threshold}):")
    for r in results:
        doc = r["document"]
        print(f"  Rank {r['rank']}: [{doc['category'].upper()}] "
              f"{doc['title']} | Score: {r['score']:.4f} | FAISS ID: {r['faiss_id']}")
    
    return results


# Test FAISS retrieval
print("🧪 Testing FAISS Retrieval...")
test_results = retrieve_with_faiss("dengue fever symptoms", top_k=3)

🧪 Testing FAISS Retrieval...

📚 FAISS Retrieved 3 Documents (threshold=0.3):
  Rank 1: [HEALTH] Dengue Fever Symptoms | Score: 0.9451 | FAISS ID: 0
  Rank 2: [HEALTH] Dengue Prevention | Score: 0.8758 | FAISS ID: 1
  Rank 3: [HEALTH] Malaria Treatment | Score: 0.8104 | FAISS ID: 2


In [15]:
def check_ollama():
    """Check if Ollama is running and show available models."""
    try:
        models = ollama.list()
        print("✅ Ollama is running!")
        print("📦 Available models:")
        for model in models.get("models", []):
            print(f"   - {model['name']}")
        return True
    except Exception as e:
        print(f"❌ Ollama not reachable: {e}")
        print("💡 Start Ollama: ollama serve")
        return False


def format_context(retrieved_docs):
    """
    Format retrieved FAISS documents into LLM context string.
    """
    context_parts = []
    
    for result in retrieved_docs:
        doc = result["document"]
        context_parts.append(
            f"[Source {result['rank']}]\n"
            f"Category: {doc['category'].upper()}\n"
            f"Title: {doc['title']}\n"
            f"Content: {doc['content']}\n"
            f"Relevance Score: {result['score']:.4f}"
        )
    
    return "\n\n---\n\n".join(context_parts)


def get_ollama_response(english_query, context, user_language):
    """
    Send query + FAISS-retrieved context to Ollama LLM.
    Always responds in English (translation handled separately).
    
    Args:
        english_query: Query translated to English
        context: Formatted context from FAISS retrieval
        user_language: Detected language code
    
    Returns:
        LLM response string in English
    """
    
    lang_name = LANGUAGE_MAP.get(user_language, "English")
    
    system_prompt = f"""You are a helpful multilingual assistant.
Answer questions based ONLY on the provided context.

Rules:
- Use ONLY the context below to answer
- If the answer is not in the context, say "I don't have information about that"
- Be clear, concise and accurate
- Always respond in ENGLISH (translation to {lang_name} is handled separately)
- Do not make up information

Context Information:
{context}
"""
    
    user_message = f"Question: {english_query}"
    
    try:
        response = ollama.chat(
            model="gemma3:4b",           # Change to your Ollama model
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_message}
            ],
            options={
                "temperature": 0.1,     # Low = more factual
                "num_predict": 512,     # Max response tokens
            }
        )
        return response["message"]["content"]
    
    except Exception as e:
        print(f"❌ Ollama error: {e}")
        return "Sorry, I could not generate a response at this time."


# Check Ollama
check_ollama()

✅ Ollama is running!
📦 Available models:
❌ Ollama not reachable: 'name'
💡 Start Ollama: ollama serve


False

In [16]:
def rag_pipeline(user_query, top_k=3, score_threshold=0.3):
    """
    Complete Trilingual RAG Pipeline with FAISS:
    
    Flow:
    ┌─────────────────────────────────────┐
    │  User Query (EN / TA / SI)          │
    │         ↓                           │
    │  Language Detection                 │
    │         ↓                           │
    │  Translate Query → English          │
    │         ↓                           │
    │  Embed Query (multilingual-e5)      │
    │         ↓                           │
    │  FAISS Vector Search                │
    │         ↓                           │
    │  Retrieve Top-K Documents           │
    │         ↓                           │
    │  Format Context                     │
    │         ↓                           │
    │  Ollama LLM (English Response)      │
    │         ↓                           │
    │  Translate Response → User Lang     │
    │         ↓                           │
    │  Final Response                     │
    └─────────────────────────────────────┘
    
    Args:
        user_query: Input in EN, TA, or SI
        top_k: FAISS documents to retrieve
        score_threshold: Min similarity score
    
    Returns:
        Final response in user's language
    """
    
    print("\n" + "=" * 60)
    print(f"💬 Query: {user_query}")
    print("=" * 60)
    
    # ── Step 1: Detect Language ────────────────────────────────
    print("\n📍 Step 1: Language Detection")
    user_lang = detect_language(user_query)
    lang_name = LANGUAGE_MAP.get(user_lang, user_lang)
    
    # ── Step 2: Translate Query to English ────────────────────
    print("\n📍 Step 2: Query Translation")
    english_query = translate_to_english(user_query, user_lang)
    
    # ── Step 3: FAISS Vector Retrieval ────────────────────────
    print("\n📍 Step 3: FAISS Vector Retrieval")
    retrieved_docs = retrieve_with_faiss(
        english_query,
        top_k=top_k,
        score_threshold=score_threshold
    )
    
    # Handle no results
    if not retrieved_docs:
        msg = "I don't have relevant information to answer your question."
        if user_lang != "en":
            msg = translate_response(msg, user_lang)
        print(f"\n⚠️ No documents found above threshold ({score_threshold})")
        return msg
    
    # ── Step 4: Format Context ────────────────────────────────
    print("\n📍 Step 4: Context Preparation")
    context = format_context(retrieved_docs)
    print(f"   ✅ Context from {len(retrieved_docs)} documents prepared")
    
    # ── Step 5: LLM Response ──────────────────────────────────
    print("\n📍 Step 5: Ollama LLM Generation")
    english_response = get_ollama_response(
        english_query,
        context,
        user_lang
    )
    print(f"   ✅ Response generated ({len(english_response)} chars)")
    
    # ── Step 6: Translate Response ────────────────────────────
    print(f"\n📍 Step 6: Translating Response → {lang_name}")
    final_response = translate_response(english_response, user_lang)
    
    # ── Final Output ──────────────────────────────────────────
    print("\n" + "=" * 60)
    print(f"🌐 Final Response [{lang_name}]:")
    print("=" * 60)
    print(final_response)
    print("=" * 60)
    
    return final_response

In [17]:
# ============================================
# TEST 1: English
# ============================================
r1 = rag_pipeline(
    "What are the symptoms of dengue fever?",
    top_k=3,
    score_threshold=0.3
)


💬 Query: What are the symptoms of dengue fever?

📍 Step 1: Language Detection

🔍 Detecting language for: 'What are the symptoms of dengue fever?'
   🔢 Unicode Analysis:
      Total chars   : 32
      Sinhala chars : 0 (0.0%)
      Tamil chars   : 0   (0.0%)
   ℹ️  No Sinhala/Tamil Unicode found → checking langdetect
   ✅ langdetect → English (en)

📍 Step 2: Query Translation
   ✅ Query already in English, no translation needed.

📍 Step 3: FAISS Vector Retrieval

📚 FAISS Retrieved 3 Documents (threshold=0.3):
  Rank 1: [HEALTH] Dengue Fever Symptoms | Score: 0.9233 | FAISS ID: 0
  Rank 2: [HEALTH] Dengue Prevention | Score: 0.8544 | FAISS ID: 1
  Rank 3: [HEALTH] Malaria Treatment | Score: 0.7950 | FAISS ID: 2

📍 Step 4: Context Preparation
   ✅ Context from 3 documents prepared

📍 Step 5: Ollama LLM Generation
   ✅ Response generated (207 chars)

📍 Step 6: Translating Response → English

🌐 Final Response [English]:
Dengue fever symptoms include high fever, severe headache, pain behind

In [18]:
# ============================================
# TEST 2: Tamil - Health
# ============================================
r2 = rag_pipeline(
    "டெங்கு காய்ச்சலை எவ்வாறு தடுக்கலாம்?",
    # "How to prevent dengue fever?"
    top_k=3,
    score_threshold=0.3
)


💬 Query: டெங்கு காய்ச்சலை எவ்வாறு தடுக்கலாம்?

📍 Step 1: Language Detection

🔍 Detecting language for: 'டெங்கு காய்ச்சலை எவ்வாறு தடுக்கலாம்?'
   🔢 Unicode Analysis:
      Total chars   : 33
      Sinhala chars : 0 (0.0%)
      Tamil chars   : 32   (97.0%)
   ✅ Unicode Detection → Tamil (ta)

📍 Step 2: Query Translation
   ⚠️ Translation failed: Response ended prematurely. Using original text.

📍 Step 3: FAISS Vector Retrieval

📚 FAISS Retrieved 3 Documents (threshold=0.3):
  Rank 1: [HEALTH] Dengue Prevention | Score: 0.8398 | FAISS ID: 1
  Rank 2: [HEALTH] Dengue Fever Symptoms | Score: 0.7853 | FAISS ID: 0
  Rank 3: [HEALTH] Malaria Treatment | Score: 0.7656 | FAISS ID: 2

📍 Step 4: Context Preparation
   ✅ Context from 3 documents prepared

📍 Step 5: Ollama LLM Generation
   ✅ Response generated (185 chars)

📍 Step 6: Translating Response → Tamil

🌐 Final Response [Tamil]:
டெங்கு காய்ச்சலைத் தடுக்க, உங்கள் வீட்டைச் சுற்றி தேங்கி நிற்கும் தண்ணீரை அகற்றவும், கொசு விரட்டியைப் பயன்படுத

In [19]:
# ============================================
# TEST 3: Sinhala - Government
# ============================================
r3 = rag_pipeline(
    "ජාතික හැඳුනුම්පත ඉල්ලුම් කරන්නේ කෙසේද?",
    # "How to apply for a national ID card?"
    top_k=3,
    score_threshold=0.3
)


💬 Query: ජාතික හැඳුනුම්පත ඉල්ලුම් කරන්නේ කෙසේද?

📍 Step 1: Language Detection

🔍 Detecting language for: 'ජාතික හැඳුනුම්පත ඉල්ලුම් කරන්නේ කෙසේද?'
   🔢 Unicode Analysis:
      Total chars   : 34
      Sinhala chars : 33 (97.1%)
      Tamil chars   : 0   (0.0%)
   ✅ Unicode Detection → Sinhala (si)

📍 Step 2: Query Translation
   🔄 Translated to English: How to apply for National Identity Card?

📍 Step 3: FAISS Vector Retrieval

📚 FAISS Retrieved 3 Documents (threshold=0.3):
  Rank 1: [GOVERNMENT] National ID Card Application | Score: 0.9161 | FAISS ID: 5
  Rank 2: [GOVERNMENT] Passport Application Process | Score: 0.8688 | FAISS ID: 6
  Rank 3: [HEALTH] COVID-19 Vaccination | Score: 0.8332 | FAISS ID: 11

📍 Step 4: Context Preparation
   ✅ Context from 3 documents prepared

📍 Step 5: Ollama LLM Generation
   ✅ Response generated (178 chars)

📍 Step 6: Translating Response → Sinhala

🌐 Final Response [Sinhala]:
ජාතික හැඳුනුම්පතක් සඳහා අයදුම් කිරීමට, ඔබ ඔබේ උප්පැන්න සහතිකය, ග්‍රාම නිලධාර

In [20]:
# ============================================
# TEST 5: Sinhala - Agriculture
# ============================================
r5 = rag_pipeline(
    "පොහොර සහනාධාරය ලබා ගන්නේ කෙසේද?",
    # "How to get fertilizer subsidy?"
    top_k=3,
    score_threshold=0.3
)


💬 Query: පොහොර සහනාධාරය ලබා ගන්නේ කෙසේද?

📍 Step 1: Language Detection

🔍 Detecting language for: 'පොහොර සහනාධාරය ලබා ගන්නේ කෙසේද?'
   🔢 Unicode Analysis:
      Total chars   : 27
      Sinhala chars : 26 (96.3%)
      Tamil chars   : 0   (0.0%)
   ✅ Unicode Detection → Sinhala (si)

📍 Step 2: Query Translation
   🔄 Translated to English: How to get Fertilizer Subsidy?

📍 Step 3: FAISS Vector Retrieval

📚 FAISS Retrieved 3 Documents (threshold=0.3):
  Rank 1: [AGRICULTURE] Fertilizer Subsidy Program | Score: 0.9191 | FAISS ID: 8
  Rank 2: [GOVERNMENT] National ID Card Application | Score: 0.7986 | FAISS ID: 5
  Rank 3: [HEALTH] COVID-19 Vaccination | Score: 0.7962 | FAISS ID: 11

📍 Step 4: Context Preparation
   ✅ Context from 3 documents prepared

📍 Step 5: Ollama LLM Generation
   ✅ Response generated (124 chars)

📍 Step 6: Translating Response → Sinhala

🌐 Final Response [Sinhala]:
සහනාධාර පොහොර ලබා ගැනීම සඳහා ගොවීන් ළඟම ඇති කෘෂිකර්ම කාර්යාලයේ ඉඩම් ඔප්පුව හෝ බදු ගිවිසුම සමඟ ලියාපදි

In [21]:
def add_documents_to_faiss(new_documents):
    """
    Add new English documents to SQLite + FAISS index.
    
    Important:
    - New docs appended to doc_store
    - New embeddings added to existing FAISS index
    - FAISS IndexFlatIP supports incremental add()
    - Index saved to disk after update
    
    Args:
        new_documents: List of (category, title, content) tuples
    
    Example:
        new_docs = [
            ("health", "Typhoid Symptoms", "Typhoid causes high fever..."),
        ]
        add_documents_to_faiss(new_docs)
    """
    
    global faiss_index, doc_store
    
    # Step 1: Insert into SQLite
    conn = sqlite3.connect("knowledge_base.db")
    cursor = conn.cursor()
    cursor.executemany(
        "INSERT INTO knowledge (category, title, content) VALUES (?, ?, ?)",
        new_documents
    )
    conn.commit()
    
    # Get newly inserted IDs
    cursor.execute(
        "SELECT id, category, title, content FROM knowledge ORDER BY id DESC LIMIT ?",
        (len(new_documents),)
    )
    new_rows = cursor.fetchall()[::-1]  # Reverse to maintain order
    conn.close()
    
    # Step 2: Build new document dicts
    new_docs = []
    for row in new_rows:
        new_docs.append({
            "db_id": row[0],
            "category": row[1],
            "title": row[2],
            "content": row[3],
            "full_text": f"passage: {row[2]}. {row[3]}"
        })
    
    # Step 3: Generate embeddings for new docs
    new_texts = [doc["full_text"] for doc in new_docs]
    new_embeddings = embedding_model.encode(
        new_texts,
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype(np.float32)
    
    # Step 4: Add to FAISS index (incremental)
    faiss_index.add(new_embeddings)
    
    # Step 5: Update doc_store
    doc_store.extend(new_docs)
    
    # Step 6: Save updated index
    save_faiss_index(faiss_index, doc_store)
    
    print(f"✅ Added {len(new_docs)} documents to FAISS")
    print(f"📊 Total vectors in index: {faiss_index.ntotal}")
    print(f"📋 Total documents in store: {len(doc_store)}")


# Example usage
example_new_docs = [
    (
        "health",
        "Diabetes Management",
        "Diabetes management includes regular blood sugar monitoring, healthy diet, physical exercise, and medication as prescribed. Patients should avoid high-sugar foods and maintain a healthy weight."
    ),
    (
        "government",
        "Driving License Application",
        "To apply for a driving license, visit the motor traffic department with national ID, medical certificate, and completed application form. You must pass written and practical driving tests."
    ),
]

# Uncomment to add:
# add_documents_to_faiss(example_new_docs)

In [22]:
def rebuild_faiss_index():
    """
    Completely rebuild FAISS index from SQLite.
    Use when:
    - Documents were deleted from SQLite
    - Index seems corrupted
    - Major data changes happened
    
    Note: FAISS IndexFlatIP does not support deletion.
    Full rebuild is the correct approach for deletions.
    """
    
    global faiss_index, doc_store, documents
    
    print("🔨 Rebuilding FAISS index from SQLite...")
    
    # Reload all documents
    documents = load_documents_from_db()
    
    # Build fresh FAISS index
    faiss_index, doc_store = build_faiss_index(documents)
    
    # Save to disk
    save_faiss_index(faiss_index, doc_store)
    
    print("✅ FAISS index rebuilt successfully!")
    print(f"📊 Total vectors: {faiss_index.ntotal}")


def delete_document_from_db(doc_id):
    """
    Delete a document from SQLite by ID.
    Triggers full FAISS rebuild since FAISS doesn't support deletion.
    
    Args:
        doc_id: SQLite row ID to delete
    """
    
    conn = sqlite3.connect("knowledge_base.db")
    cursor = conn.cursor()
    cursor.execute("DELETE FROM knowledge WHERE id = ?", (doc_id,))
    conn.commit()
    conn.close()
    
    print(f"✅ Document {doc_id} deleted from SQLite")
    print("🔨 Rebuilding FAISS index (required after deletion)...")
    
    # Must rebuild since FAISS doesn't support deletion
    rebuild_faiss_index()


# View all documents before deleting
def view_database():
    """Display all documents in SQLite."""
    conn = sqlite3.connect("knowledge_base.db")
    cursor = conn.cursor()
    cursor.execute("SELECT id, category, title FROM knowledge ORDER BY category")
    rows = cursor.fetchall()
    conn.close()
    
    print(f"\n📊 SQLite Database ({len(rows)} documents)")
    print("=" * 55)
    
    current_cat = None
    for row in rows:
        if row[1] != current_cat:
            current_cat = row[1]
            print(f"\n📁 {current_cat.upper()}")
            print("-" * 40)
        print(f"  [ID:{row[0]}] {row[2]}")
    
    print()

view_database()

# Uncomment to delete:
# delete_document_from_db(1)


📊 SQLite Database (12 documents)

📁 AGRICULTURE
----------------------------------------
  [ID:8] Paddy Cultivation Season
  [ID:9] Fertilizer Subsidy Program

📁 EDUCATION
----------------------------------------
  [ID:4] School Admission Process
  [ID:5] Scholarship Eligibility

📁 FINANCE
----------------------------------------
  [ID:10] Bank Loan Application
  [ID:11] Samurdhi Benefits

📁 GOVERNMENT
----------------------------------------
  [ID:6] National ID Card Application
  [ID:7] Passport Application Process

📁 HEALTH
----------------------------------------
  [ID:1] Dengue Fever Symptoms
  [ID:2] Dengue Prevention
  [ID:3] Malaria Treatment
  [ID:12] COVID-19 Vaccination



In [23]:
%%writefile app.py

# ============================================================
# TRILINGUAL RAG BOT
# English + Tamil + Sinhala
# FAISS + SQLite + Ollama + Streamlit
# FULLY OFFLINE — LLM handles all translation
# ============================================================

import streamlit as st
import sqlite3
import numpy as np
import os
import pickle
import warnings
import faiss
import time
import json
import requests
from datetime import datetime
from collections import Counter

warnings.filterwarnings("ignore")
os.environ["HF_HUB_DISABLE_XET"] = "1"

from sentence_transformers import SentenceTransformer
from langdetect import detect, DetectorFactory

DetectorFactory.seed = 0

# ============================================================
# CONFIGURATION
# ============================================================

CONFIG = {
    "db_path"          : "knowledge_base.db",
    "faiss_index_path" : "faiss_index.bin",
    "doc_store_path"   : "doc_store.pkl",
    "embedding_model"  : "intfloat/multilingual-e5-small",
    "ollama_model"     : "gemma3:12b",
    "ollama_base_url"  : "http://localhost:11434",   # REST endpoint
    "top_k"            : 3,
    "score_threshold"  : 0.3,
    "temperature"      : 0.1,
    "max_tokens"       : 512,
}

LANGUAGE_MAP = {
    "en": "English",
    "ta": "Tamil   (தமிழ்)",
    "si": "Sinhala (සිංහල)"
}

LANGUAGE_FULL = {
    "en": "English",
    "ta": "Tamil",
    "si": "Sinhala"
}

LANGUAGE_FLAGS = {
    "en": "🇬🇧",
    "ta": "🇮🇳",
    "si": "🇱🇰"
}

# Unicode ranges
SINHALA_START = 0x0D80
SINHALA_END   = 0x0DFF
TAMIL_START   = 0x0B80
TAMIL_END     = 0x0BFF

# ============================================================
# STREAMLIT PAGE CONFIG
# ============================================================

st.set_page_config(
    page_title="Trilingual RAG Bot",
    page_icon="🌐",
    layout="wide",
    initial_sidebar_state="expanded"
)

# ============================================================
# CUSTOM CSS
# ============================================================

st.markdown("""
<style>
    .main { background-color: #0e1117; }

    .header-banner {
        background: linear-gradient(135deg, #1a1a2e 0%, #16213e 50%, #0f3460 100%);
        padding: 30px; border-radius: 15px; text-align: center;
        margin-bottom: 25px; border: 1px solid #e94560;
        box-shadow: 0 4px 20px rgba(233,69,96,0.3);
    }
    .header-title {
        font-size: 2.5rem; font-weight: 800; color: #ffffff;
        margin: 0; letter-spacing: 2px;
    }
    .header-subtitle {
        font-size: 1rem; color: #a0aec0; margin-top: 8px; letter-spacing: 1px;
    }
    .lang-badges {
        margin-top: 15px; display: flex; justify-content: center;
        gap: 15px; flex-wrap: wrap;
    }
    .lang-badge {
        background: rgba(233,69,96,0.2); border: 1px solid #e94560;
        color: #e94560; padding: 5px 15px; border-radius: 20px;
        font-size: 0.85rem; font-weight: 600;
    }

    .chat-user {
        background: linear-gradient(135deg, #1e3a5f, #0f3460);
        border: 1px solid #2d6a9f; border-radius: 15px 15px 5px 15px;
        padding: 15px 20px; margin: 10px 0; color: #e2e8f0;
        max-width: 80%; margin-left: auto;
        box-shadow: 0 2px 10px rgba(45,106,159,0.3);
    }
    .chat-bot {
        background: linear-gradient(135deg, #1a2e1a, #1e3a1e);
        border: 1px solid #2d7a2d; border-radius: 15px 15px 15px 5px;
        padding: 15px 20px; margin: 10px 0; color: #e2e8f0;
        max-width: 80%; box-shadow: 0 2px 10px rgba(45,122,45,0.3);
    }
    .chat-label-user {
        font-size: 0.75rem; color: #63b3ed; font-weight: 700;
        margin-bottom: 6px; text-transform: uppercase; letter-spacing: 1px;
    }
    .chat-label-bot {
        font-size: 0.75rem; color: #68d391; font-weight: 700;
        margin-bottom: 6px; text-transform: uppercase; letter-spacing: 1px;
    }
    .chat-meta {
        font-size: 0.70rem; color: #718096; margin-top: 8px; text-align: right;
    }

    .retrieved-doc {
        background: #1a1a2e; border: 1px solid #2d3748;
        border-left: 4px solid #e94560; border-radius: 8px;
        padding: 12px 15px; margin: 8px 0; font-size: 0.85rem;
    }
    .doc-title { color: #e94560; font-weight: 700; font-size: 0.9rem; }
    .doc-category {
        background: rgba(233,69,96,0.15); color: #fc8181;
        padding: 2px 8px; border-radius: 10px;
        font-size: 0.75rem; font-weight: 600;
    }
    .doc-score { color: #68d391; font-weight: 700; }
    .doc-content { color: #a0aec0; margin-top: 6px; line-height: 1.5; }

    .metric-card {
        background: linear-gradient(135deg, #1a1a2e, #16213e);
        border: 1px solid #2d3748; border-radius: 12px;
        padding: 20px; text-align: center; margin: 5px 0;
    }
    .metric-value { font-size: 2rem; font-weight: 800; color: #e94560; }
    .metric-label {
        font-size: 0.8rem; color: #718096;
        text-transform: uppercase; letter-spacing: 1px; margin-top: 5px;
    }

    .status-ok    { color: #68d391; font-weight: 600; }
    .status-error { color: #fc8181; font-weight: 600; }
    .status-warn  { color: #f6ad55; font-weight: 600; }

    .lang-detect-box {
        background: linear-gradient(135deg, #1a2e1a, #2d3748);
        border: 1px solid #4a5568; border-radius: 10px;
        padding: 12px 15px; margin: 8px 0;
        font-size: 0.85rem; color: #a0aec0;
    }
    .pipeline-step {
        background: #1a1a2e; border: 1px solid #2d3748;
        border-radius: 8px; padding: 10px 15px; margin: 4px 0;
        font-size: 0.82rem; color: #a0aec0;
        display: flex; align-items: center; gap: 10px;
    }
    .unicode-box {
        background: #0d1117; border: 1px solid #30363d;
        border-radius: 8px; padding: 10px;
        font-family: monospace; font-size: 0.8rem; color: #8b949e;
    }
    .section-header {
        font-size: 1rem; font-weight: 700; color: #e94560;
        text-transform: uppercase; letter-spacing: 2px;
        margin: 20px 0 10px 0; padding-bottom: 5px;
        border-bottom: 1px solid #e94560;
    }
    .custom-divider {
        height: 1px;
        background: linear-gradient(90deg, transparent, #e94560, transparent);
        margin: 20px 0;
    }

    .stTextInput > div > div > input {
        background-color: #1a1a2e !important; border: 1px solid #4a5568 !important;
        color: #e2e8f0 !important; border-radius: 10px !important;
        padding: 12px !important; font-size: 1rem !important;
    }
    .stButton > button {
        background: linear-gradient(135deg, #e94560, #c0392b) !important;
        color: white !important; border: none !important;
        border-radius: 10px !important; padding: 10px 25px !important;
        font-weight: 700 !important; font-size: 0.9rem !important;
        transition: all 0.3s ease !important; width: 100% !important;
    }
    .stButton > button:hover {
        transform: translateY(-2px) !important;
        box-shadow: 0 5px 15px rgba(233,69,96,0.4) !important;
    }

    #MainMenu {visibility: hidden;}
    footer {visibility: hidden;}
    header {visibility: hidden;}
</style>
""", unsafe_allow_html=True)


# ============================================================
# OLLAMA — REST-BASED HELPERS  (fixes "always offline" bug)
# ============================================================
# The `ollama` Python package's .list() call fails in many
# environments because it defaults to a different port or
# raises on unexpected response shapes.  We talk directly to
# the Ollama REST API instead, which is always reliable.

def ollama_get(path, timeout=3):
    """GET from Ollama REST API."""
    url = f"{CONFIG['ollama_base_url']}{path}"
    r   = requests.get(url, timeout=timeout)
    r.raise_for_status()
    return r.json()


def ollama_post(path, payload, timeout=120):
    """POST to Ollama REST API."""
    url = f"{CONFIG['ollama_base_url']}{path}"
    r   = requests.post(url, json=payload, timeout=timeout)
    r.raise_for_status()
    return r.json()


def check_ollama_status():
    """
    Check Ollama availability via REST.
    Returns (is_running: bool, model_names: list[str]).
    """
    try:
        data   = ollama_get("/api/tags", timeout=3)
        models = data.get("models", [])
        names  = [m.get("name", "") for m in models]
        return True, names
    except Exception:
        return False, []


def ollama_chat(model_name, messages, temperature=0.1, max_tokens=512):
    """
    Call Ollama /api/chat and return (content, error).
    Uses streaming=False for simplicity.
    """
    payload = {
        "model"   : model_name,
        "messages": messages,
        "stream"  : False,
        "options" : {
            "temperature": temperature,
            "num_predict": max_tokens,
        },
    }
    try:
        data    = ollama_post("/api/chat", payload, timeout=180)
        content = data["message"]["content"]
        return content, None
    except requests.exceptions.ConnectionError:
        return None, "Cannot connect to Ollama. Make sure `ollama serve` is running."
    except requests.exceptions.Timeout:
        return None, "Ollama request timed out."
    except Exception as e:
        return None, str(e)


# ============================================================
# UNICODE LANGUAGE DETECTION
# ============================================================

def count_script_characters(text, start, end):
    return sum(1 for ch in text if start <= ord(ch) <= end)


def detect_language_unicode(text):
    clean_text = text.strip()
    if not clean_text:
        return "en", 0, 0, 0, 0.0, 0.0

    sinhala_count = count_script_characters(clean_text, SINHALA_START, SINHALA_END)
    tamil_count   = count_script_characters(clean_text, TAMIL_START,   TAMIL_END)
    total_chars   = len(clean_text.replace(" ", ""))

    sinhala_ratio = sinhala_count / total_chars if total_chars > 0 else 0.0
    tamil_ratio   = tamil_count   / total_chars if total_chars > 0 else 0.0

    THRESHOLD = 0.10
    if sinhala_ratio >= THRESHOLD:
        lang = "si"
    elif tamil_ratio >= THRESHOLD:
        lang = "ta"
    else:
        lang = "en"

    return lang, sinhala_count, tamil_count, total_chars, sinhala_ratio, tamil_ratio


def detect_language_langdetect(text):
    try:
        return detect(text)
    except Exception:
        return "en"


def detect_language(text):
    """
    Hybrid detection:
    1. Unicode scan  → catches Sinhala / Tamil perfectly
    2. langdetect    → English confirmation fallback
    Returns (lang_code, details_dict)
    """
    (unicode_lang,
     sinhala_count, tamil_count,
     total_chars,
     sinhala_ratio, tamil_ratio) = detect_language_unicode(text)

    details = {
        "method"         : "Unicode",
        "sinhala_chars"  : sinhala_count,
        "tamil_chars"    : tamil_count,
        "total_chars"    : total_chars,
        "sinhala_ratio"  : sinhala_ratio,
        "tamil_ratio"    : tamil_ratio,
        "unicode_result" : unicode_lang,
        "fallback_result": None,
        "final_lang"     : unicode_lang,
    }

    if unicode_lang in ["si", "ta"]:
        details["method"] = "Unicode (Primary)"
        return unicode_lang, details

    fallback = detect_language_langdetect(text)
    details["fallback_result"] = fallback
    details["method"]          = "langdetect (Fallback)"

    if fallback in ["en", "ta", "si"]:
        details["final_lang"] = fallback
        return fallback, details

    details["final_lang"] = "en"
    return "en", details


# ============================================================
# LLM-BASED TRANSLATION  (fully offline, no Google Translate)
# ============================================================

def translate_to_english_llm(text, source_lang, model_name):
    """
    Ask Ollama to translate Tamil/Sinhala text into English.
    Returns (translated_text, info_string).
    """
    if source_lang == "en":
        return text, "No translation needed"

    lang_name = LANGUAGE_FULL.get(source_lang, source_lang)

    system = (
        "You are a professional translator. "
        "Translate the user's text from "
        f"{lang_name} to English. "
        "Output ONLY the English translation — "
        "no explanations, no notes, no punctuation wrappers."
    )

    content, error = ollama_chat(
        model_name,
        [
            {"role": "system", "content": system},
            {"role": "user",   "content": text},
        ],
        temperature=0.1,
        max_tokens=256,
    )

    if error:
        # Graceful degradation: pass the original text and warn
        return text, f"Translation failed ({error}) — using original text"

    translated = content.strip()
    return translated, translated


def translate_response_llm(text, target_lang, model_name):
    """
    Ask Ollama to translate an English response into the target language.
    Returns translated string (falls back to English on error).
    """
    if target_lang == "en":
        return text

    lang_name = LANGUAGE_FULL.get(target_lang, target_lang)

    system = (
        "You are a professional translator. "
        f"Translate the following English text into {lang_name}. "
        "Output ONLY the translation — no extra commentary."
        "Avoid mixing languages."
    )

    content, error = ollama_chat(
        model_name,
        [
            {"role": "system", "content": system},
            {"role": "user",   "content": text},
        ],
        temperature=0.1,
        max_tokens=700,
    )

    if error:
        return text   # Fall back to English if translation fails

    return content.strip()


# ============================================================
# DATABASE SETUP
# ============================================================

def create_database():
    conn   = sqlite3.connect(CONFIG["db_path"])
    cursor = conn.cursor()

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS knowledge (
            id       INTEGER PRIMARY KEY AUTOINCREMENT,
            category TEXT NOT NULL,
            title    TEXT NOT NULL,
            content  TEXT NOT NULL
        )
    """)

    cursor.execute("SELECT COUNT(*) FROM knowledge")
    count = cursor.fetchone()[0]

    if count == 0:
        sample_data = [
            ("health",
             "Dengue Fever Symptoms",
             "Dengue fever symptoms include high fever, severe headache, pain behind the eyes, "
             "joint and muscle pain, rash, and mild bleeding. Symptoms usually appear 4 to 10 "
             "days after infection and last for 2 to 7 days."),

            ("health",
             "Dengue Prevention",
             "To prevent dengue fever, eliminate standing water around your home, use mosquito "
             "repellent, wear long-sleeved clothing, use mosquito nets, and keep windows and "
             "doors closed or screened."),

            ("health",
             "Malaria Treatment",
             "Malaria treatment depends on the type of malaria parasite and severity. Common "
             "treatments include antimalarial medications such as chloroquine, artemisinin-based "
             "combination therapies, and supportive care."),

            ("education",
             "School Admission Process",
             "School admission requires birth certificate, vaccination records, previous school "
             "records if applicable, and parent identification documents. Applications must be "
             "submitted before the deadline."),

            ("education",
             "Scholarship Eligibility",
             "Scholarships are available for students with academic excellence, financial need, "
             "or special talents. Students must maintain a minimum GPA of 3.5 and submit "
             "applications by March 31 each year."),

            ("government",
             "National ID Card Application",
             "To apply for a national ID card, you need to visit the nearest divisional "
             "secretariat with your birth certificate, Grama Niladhari certificate, and two "
             "passport-size photographs."),

            ("government",
             "Passport Application Process",
             "Passport application requires national ID card, birth certificate, police clearance "
             "report, and completed application form. Processing takes 3 to 10 working days for "
             "normal service."),

            ("agriculture",
             "Paddy Cultivation Season",
             "Sri Lanka has two main paddy cultivation seasons. Maha season runs from October to "
             "March and Yala season runs from April to September. Farmers receive government "
             "subsidies for seeds and fertilizer."),

            ("agriculture",
             "Fertilizer Subsidy Program",
             "The government provides fertilizer subsidies to registered farmers. Farmers must "
             "register at the nearest agricultural office with land deed or lease agreement to "
             "receive subsidized fertilizer."),

            ("finance",
             "Bank Loan Application",
             "To apply for a bank loan, you need income proof, bank statements for last 6 months,"
             " national ID, and collateral documents if applicable. Loan approval typically takes "
             "5 to 7 working days."),

            ("finance",
             "Samurdhi Benefits",
             "Samurdhi is a government welfare program providing financial assistance to "
             "low-income families. Eligible families receive monthly allowances and can access "
             "low-interest loans through Samurdhi banks."),

            ("health",
             "COVID-19 Vaccination",
             "COVID-19 vaccines are available at government hospitals and health centers. Bring "
             "your national ID card for registration. Booster doses are recommended every 6 "
             "months for high-risk groups."),

            ("health",
             "Diabetes Management",
             "Diabetes management includes regular blood sugar monitoring, healthy diet, physical "
             "exercise, and medication as prescribed. Patients should avoid high-sugar foods and "
             "maintain a healthy weight."),

            ("government",
             "Driving License Application",
             "To apply for a driving license, visit the motor traffic department with national ID,"
             " medical certificate, and completed application form. You must pass written and "
             "practical driving tests."),
        ]

        cursor.executemany(
            "INSERT INTO knowledge (category, title, content) VALUES (?, ?, ?)",
            sample_data
        )
        conn.commit()

    conn.close()


def load_documents_from_db():
    conn   = sqlite3.connect(CONFIG["db_path"])
    cursor = conn.cursor()
    cursor.execute("SELECT id, category, title, content FROM knowledge")
    rows   = cursor.fetchall()
    conn.close()
    return [
        {
            "db_id"    : row[0],
            "category" : row[1],
            "title"    : row[2],
            "content"  : row[3],
            "full_text": f"passage: {row[2]}. {row[3]}"
        }
        for row in rows
    ]


def get_db_stats():
    conn   = sqlite3.connect(CONFIG["db_path"])
    cursor = conn.cursor()
    cursor.execute("SELECT category, COUNT(*) FROM knowledge GROUP BY category")
    cats   = cursor.fetchall()
    cursor.execute("SELECT COUNT(*) FROM knowledge")
    total  = cursor.fetchone()[0]
    conn.close()
    return total, dict(cats)


def get_all_documents_display():
    conn   = sqlite3.connect(CONFIG["db_path"])
    cursor = conn.cursor()
    cursor.execute("SELECT id, category, title, content FROM knowledge ORDER BY category, id")
    rows   = cursor.fetchall()
    conn.close()
    return rows


def add_document_to_db(category, title, content):
    conn   = sqlite3.connect(CONFIG["db_path"])
    cursor = conn.cursor()
    cursor.execute(
        "INSERT INTO knowledge (category, title, content) VALUES (?, ?, ?)",
        (category, title, content)
    )
    new_id = cursor.lastrowid
    conn.commit()
    conn.close()
    return new_id


def delete_document_from_db(doc_id):
    conn   = sqlite3.connect(CONFIG["db_path"])
    cursor = conn.cursor()
    cursor.execute("DELETE FROM knowledge WHERE id = ?", (doc_id,))
    conn.commit()
    conn.close()


# ============================================================
# EMBEDDING MODEL (cached)
# ============================================================

@st.cache_resource(show_spinner=False)
def load_embedding_model():
    return SentenceTransformer(CONFIG["embedding_model"])


# ============================================================
# FAISS INDEX
# ============================================================

def build_faiss_index(documents, model):
    texts      = [doc["full_text"] for doc in documents]
    embeddings = model.encode(
        texts,
        normalize_embeddings=True,
        show_progress_bar=False,
        batch_size=32,
        convert_to_numpy=True
    ).astype(np.float32)

    dim   = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings)
    return index, documents


def save_faiss_index(index, doc_store):
    faiss.write_index(index, CONFIG["faiss_index_path"])
    with open(CONFIG["doc_store_path"], "wb") as f:
        pickle.dump(doc_store, f)


def load_faiss_index_from_disk():
    if (os.path.exists(CONFIG["faiss_index_path"]) and
            os.path.exists(CONFIG["doc_store_path"])):
        index = faiss.read_index(CONFIG["faiss_index_path"])
        with open(CONFIG["doc_store_path"], "rb") as f:
            doc_store = pickle.load(f)
        return index, doc_store
    return None, None


@st.cache_resource(show_spinner=False)
def initialize_faiss(_model):
    create_database()
    index, doc_store = load_faiss_index_from_disk()
    if index is None:
        documents        = load_documents_from_db()
        index, doc_store = build_faiss_index(documents, _model)
        save_faiss_index(index, doc_store)
    return index, doc_store


def rebuild_index(_model):
    documents        = load_documents_from_db()
    index, doc_store = build_faiss_index(documents, _model)
    save_faiss_index(index, doc_store)
    return index, doc_store


# ============================================================
# FAISS RETRIEVAL
# ============================================================

def retrieve_with_faiss(query_english, model, index, doc_store,
                        top_k=3, score_threshold=0.3):
    query_emb = model.encode(
        [f"query: {query_english}"],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype(np.float32)

    distances, indices = index.search(query_emb, top_k)

    results = []
    for rank, (score, idx) in enumerate(zip(distances[0], indices[0])):
        if idx == -1:
            continue
        if score < score_threshold:
            continue
        results.append({
            "document": doc_store[idx],
            "score"   : float(score),
            "faiss_id": int(idx),
            "rank"    : rank + 1
        })
    return results


# ============================================================
# LLM — ANSWER GENERATION
# ============================================================

def format_context(retrieved_docs):
    parts = []
    for r in retrieved_docs:
        doc = r["document"]
        parts.append(
            f"[Source {r['rank']}]\n"
            f"Category : {doc['category'].upper()}\n"
            f"Title    : {doc['title']}\n"
            f"Content  : {doc['content']}\n"
            f"Score    : {r['score']:.4f}"
        )
    return "\n\n---\n\n".join(parts)


def get_ollama_response(english_query, context, model_name):
    """Generate a grounded English answer from retrieved context."""
    system_prompt = f"""You are a helpful assistant.
Answer the question based ONLY on the provided context.

Rules:
- Use ONLY the context below to answer.
- If the answer is not in the context, say exactly:
  "I don't have information about that."
- Be clear, concise and accurate.
- Always respond in ENGLISH.
- Do not make up information.

Context:
{context}
"""
    return ollama_chat(
        model_name,
        [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": f"Question: {english_query}"},
        ],
        temperature=CONFIG["temperature"],
        max_tokens=CONFIG["max_tokens"],
    )


# ============================================================
# FULL RAG PIPELINE
# ============================================================

def run_rag_pipeline(user_query, model, faiss_index, doc_store,
                     top_k, score_threshold, ollama_model):
    result = {
        "query"           : user_query,
        "timestamp"       : datetime.now().strftime("%H:%M:%S"),
        "lang_code"       : "en",
        "lang_name"       : "English",
        "lang_details"    : {},
        "english_query"   : user_query,
        "translation_info": "",
        "retrieved_docs"  : [],
        "context"         : "",
        "english_response": "",
        "final_response"  : "",
        "error"           : None,
        "timings"         : {}
    }

    # ── Step 1: Language Detection ────────────────────────────
    t0                      = time.time()
    lang_code, lang_details = detect_language(user_query)
    result["lang_code"]     = lang_code
    result["lang_name"]     = LANGUAGE_MAP.get(lang_code, "English")
    result["lang_details"]  = lang_details
    result["timings"]["lang_detection"] = round(time.time() - t0, 3)

    # ── Step 2: Translate query → English (via LLM) ───────────
    t0                         = time.time()
    english_query, trans_info  = translate_to_english_llm(
        user_query, lang_code, ollama_model
    )
    result["english_query"]    = english_query
    result["translation_info"] = trans_info
    result["timings"]["translation"] = round(time.time() - t0, 3)

    # ── Step 3: FAISS Retrieval ───────────────────────────────
    t0                       = time.time()
    retrieved                = retrieve_with_faiss(
        english_query, model, faiss_index, doc_store,
        top_k=top_k, score_threshold=score_threshold
    )
    result["retrieved_docs"] = retrieved
    result["timings"]["retrieval"] = round(time.time() - t0, 3)

    if not retrieved:
        msg                      = "I don't have relevant information to answer your question."
        result["english_response"] = msg
        if lang_code != "en":
            t0 = time.time()
            result["final_response"] = translate_response_llm(msg, lang_code, ollama_model)
            result["timings"]["response_translation"] = round(time.time() - t0, 3)
        else:
            result["final_response"] = msg
        return result

    # ── Step 4: Format Context ────────────────────────────────
    result["context"] = format_context(retrieved)

    # ── Step 5: LLM Answer (English) ─────────────────────────
    t0                         = time.time()
    eng_response, error        = get_ollama_response(
        english_query, result["context"], ollama_model
    )
    result["timings"]["llm"]   = round(time.time() - t0, 3)

    if error:
        result["error"]          = error
        result["final_response"] = f"❌ Ollama Error: {error}"
        return result

    result["english_response"] = eng_response

    # ── Step 6: Translate Response → user language (via LLM) ─
    t0 = time.time()
    result["final_response"] = translate_response_llm(
        eng_response, lang_code, ollama_model
    )
    result["timings"]["response_translation"] = round(time.time() - t0, 3)

    return result


# ============================================================
# STREAMLIT SESSION STATE INIT
# ============================================================

def init_session_state():
    defaults = {
        "chat_history"  : [],
        "faiss_index"   : None,
        "doc_store"     : None,
        "last_result"   : None,
        "total_queries" : 0,
        "lang_counter"  : {"en": 0, "ta": 0, "si": 0},
    }
    for k, v in defaults.items():
        if k not in st.session_state:
            st.session_state[k] = v


# ============================================================
# SIDEBAR
# ============================================================

def render_sidebar(model):
    with st.sidebar:

        st.markdown("""
        <div style='text-align:center; padding: 10px 0 20px 0;'>
            <div style='font-size:2.5rem;'>🌐</div>
            <div style='font-size:1.1rem; font-weight:800;
                        color:#e94560; letter-spacing:2px;'>RAG BOT</div>
            <div style='font-size:0.75rem; color:#718096;'>Trilingual Intelligence</div>
        </div>
        """, unsafe_allow_html=True)

        st.markdown("---")

        # ── System Status ─────────────────────────────────────
        st.markdown('<div class="section-header">⚙️ System Status</div>',
                    unsafe_allow_html=True)

        st.markdown(
            '<span class="status-ok">✅ Embedding Model Ready</span><br>'
            f'<small style="color:#718096;">{CONFIG["embedding_model"]}</small>',
            unsafe_allow_html=True
        )

        if st.session_state.faiss_index is not None:
            n = st.session_state.faiss_index.ntotal
            st.markdown(
                f'<span class="status-ok">✅ FAISS Index ({n} vectors)</span>',
                unsafe_allow_html=True
            )
        else:
            st.markdown(
                '<span class="status-warn">⏳ Building FAISS Index...</span>',
                unsafe_allow_html=True
            )

        ollama_ok, ollama_models = check_ollama_status()
        if ollama_ok:
            st.markdown(
                '<span class="status-ok">✅ Ollama Running</span>',
                unsafe_allow_html=True
            )
        else:
            st.markdown(
                '<span class="status-error">❌ Ollama Offline — run: ollama serve</span>',
                unsafe_allow_html=True
            )

        st.markdown(
            '<span style="color:#a0aec0; font-size:0.78rem;">'
            '🔒 Fully Offline — no Google Translate</span>',
            unsafe_allow_html=True
        )

        st.markdown("---")

        # ── Settings ──────────────────────────────────────────
        st.markdown('<div class="section-header">🎛️ Settings</div>',
                    unsafe_allow_html=True)

        if ollama_ok and ollama_models:
            selected_model = st.selectbox("🤖 Ollama Model", options=ollama_models, index=0)
        else:
            selected_model = st.text_input("🤖 Ollama Model", value=CONFIG["ollama_model"])

        top_k = st.slider(
            "📚 Retrieved Documents (Top-K)",
            min_value=1, max_value=6, value=CONFIG["top_k"],
            help="Number of documents to retrieve from FAISS"
        )

        score_threshold = st.slider(
            "🎯 Similarity Threshold",
            min_value=0.0, max_value=1.0,
            value=CONFIG["score_threshold"], step=0.05,
            help="Minimum cosine similarity score"
        )

        temperature = st.slider(
            "🌡️ LLM Temperature",
            min_value=0.0, max_value=1.0,
            value=CONFIG["temperature"], step=0.05,
            help="Lower = more factual"
        )
        CONFIG["temperature"] = temperature

        st.markdown("---")

        # ── Language Stats ────────────────────────────────────
        st.markdown('<div class="section-header">🌐 Language Stats</div>',
                    unsafe_allow_html=True)

        lc   = st.session_state.lang_counter
        cols = st.columns(3)
        with cols[0]: st.metric("🇬🇧 EN", lc.get("en", 0))
        with cols[1]: st.metric("🇮🇳 TA", lc.get("ta", 0))
        with cols[2]: st.metric("🇱🇰 SI", lc.get("si", 0))
        st.caption(f"Total queries: {st.session_state.total_queries}")

        st.markdown("---")

        # ── Database Stats ────────────────────────────────────
        st.markdown('<div class="section-header">🗄️ Database</div>',
                    unsafe_allow_html=True)

        total_docs, cat_stats = get_db_stats()
        st.caption(f"Total documents: {total_docs}")

        for cat, cnt in sorted(cat_stats.items()):
            emoji = {
                "health": "🏥", "education": "🎓",
                "government": "🏛️", "agriculture": "🌾", "finance": "💰"
            }.get(cat, "📄")
            st.markdown(
                f'<div style="display:flex; justify-content:space-between; '
                f'font-size:0.82rem; color:#a0aec0; padding:2px 0;">'
                f'<span>{emoji} {cat.capitalize()}</span>'
                f'<span style="color:#e94560; font-weight:700;">{cnt}</span></div>',
                unsafe_allow_html=True
            )

        st.markdown("---")

        # ── Maintenance ───────────────────────────────────────
        st.markdown('<div class="section-header">🔧 Maintenance</div>',
                    unsafe_allow_html=True)

        if st.button("🔄 Rebuild FAISS Index"):
            with st.spinner("Rebuilding..."):
                idx, ds = rebuild_index(model)
                st.session_state.faiss_index = idx
                st.session_state.doc_store   = ds
                initialize_faiss.clear()
            st.success(f"✅ Index rebuilt! ({idx.ntotal} vectors)")
            st.rerun()

        if st.button("🗑️ Clear Chat History"):
            st.session_state.chat_history  = []
            st.session_state.last_result   = None
            st.session_state.total_queries = 0
            st.session_state.lang_counter  = {"en": 0, "ta": 0, "si": 0}
            st.rerun()

        st.markdown("---")
        st.caption("🔒 Fully Local | No Cloud API")
        st.caption("FAISS + SQLite + Ollama")

    return selected_model, top_k, score_threshold


# ============================================================
# CHAT TAB
# ============================================================

def render_chat_tab(model, faiss_index, doc_store,
                    top_k, score_threshold, ollama_model):

    # ── Example Queries ───────────────────────────────────────
    st.markdown('<div class="section-header">💡 Example Queries</div>',
                unsafe_allow_html=True)

    examples = {
        "🇬🇧 English": [
            "What are the symptoms of dengue fever?",
            "How to apply for a passport?",
            "What is Samurdhi benefit program?",
        ],
        "🇮🇳 Tamil": [
            "டெங்கு காய்ச்சலின் அறிகுறிகள் என்ன?",
            "வங்கி கடன் விண்ணப்பிக்க என்ன தேவை?",
            "கல்வி உதவித்தொகை எப்படி பெறுவது?",
        ],
        "🇱🇰 Sinhala": [
            "ඩෙංගු රෝග ලක්ෂණ මොනවාද?",
            "ජාතික හැඳුනුම්පත ඉල්ලුම් කරන්නේ කෙසේද?",
            "පොහොර සහනාධාරය ලබා ගන්නේ කෙසේද?",
        ],
    }

    ex_cols = st.columns(3)
    example_clicked = None

    for col, (lang_label, queries) in zip(ex_cols, examples.items()):
        with col:
            st.caption(lang_label)
            for q in queries:
                label = q[:35] + "..." if len(q) > 35 else q
                if st.button(label, key=f"ex_{q[:20]}", use_container_width=True):
                    example_clicked = q

    st.markdown('<div class="custom-divider"></div>', unsafe_allow_html=True)

    # ── Chat History ──────────────────────────────────────────
    st.markdown('<div class="section-header">💬 Conversation</div>',
                unsafe_allow_html=True)

    if not st.session_state.chat_history:
        st.markdown("""
        <div style='text-align:center; padding:40px; color:#4a5568;'>
            <div style='font-size:3rem;'>🌐</div>
            <div style='font-size:1.1rem; margin-top:10px;'>
                Ask anything in English, Tamil, or Sinhala
            </div>
            <div style='font-size:0.85rem; margin-top:5px; color:#2d3748;'>
                ඕනෑම භාෂාවෙන් අසන්න | எந்த மொழியிலும் கேளுங்கள்
            </div>
        </div>
        """, unsafe_allow_html=True)
    else:
        for chat in st.session_state.chat_history:
            flag = LANGUAGE_FLAGS.get(chat.get("lang_code", "en"), "🌐")
            total_time = sum(chat.get("timings", {}).values())

            st.markdown(f"""
            <div class="chat-user">
                <div class="chat-label-user">{flag} You · {chat.get('timestamp','')}</div>
                <div>{chat['query']}</div>
                <div class="chat-meta">{LANGUAGE_MAP.get(chat.get('lang_code','en'), 'English')}</div>
            </div>
            """, unsafe_allow_html=True)

            st.markdown(f"""
            <div class="chat-bot">
                <div class="chat-label-bot">🤖 RAG Bot · {chat.get('timestamp','')}</div>
                <div>{chat['response']}</div>
                <div class="chat-meta">
                    ⏱️ {total_time:.2f}s |
                    📚 {len(chat.get('retrieved_docs', []))} docs retrieved
                </div>
            </div>
            """, unsafe_allow_html=True)

    st.markdown('<div class="custom-divider"></div>', unsafe_allow_html=True)

    # ── Input Area ────────────────────────────────────────────
    input_col, btn_col = st.columns([5, 1])

    with input_col:
        default_val = example_clicked if example_clicked else ""
        user_input  = st.text_input(
            "Your question",
            value=default_val,
            placeholder="Type in English, Tamil (தமிழ்) or Sinhala (සිංහල)...",
            label_visibility="collapsed",
            key="main_input"
        )

    with btn_col:
        send_clicked = st.button("Send 🚀", use_container_width=True)

    # ── Process ───────────────────────────────────────────────
    query_to_run = None
    if send_clicked and user_input.strip():
        query_to_run = user_input.strip()
    elif example_clicked:
        query_to_run = example_clicked

    if query_to_run:
        with st.spinner("🔍 Processing your query..."):
            result = run_rag_pipeline(
                query_to_run, model, faiss_index, doc_store,
                top_k, score_threshold, ollama_model
            )

        st.session_state.chat_history.append({
            "query"         : query_to_run,
            "response"      : result["final_response"],
            "lang_code"     : result["lang_code"],
            "timestamp"     : result["timestamp"],
            "retrieved_docs": result["retrieved_docs"],
            "timings"       : result["timings"],
        })

        st.session_state.total_queries += 1
        lc = st.session_state.lang_counter
        lc[result["lang_code"]] = lc.get(result["lang_code"], 0) + 1
        st.session_state.last_result = result
        st.rerun()


# ============================================================
# PIPELINE INSPECTOR TAB
# ============================================================

def render_inspector_tab():
    st.markdown('<div class="section-header">🔬 Pipeline Inspector</div>',
                unsafe_allow_html=True)

    result = st.session_state.last_result
    if result is None:
        st.info("💡 Run a query first to inspect the pipeline.")
        return

    flag      = LANGUAGE_FLAGS.get(result["lang_code"], "🌐")
    lang_name = LANGUAGE_MAP.get(result["lang_code"], "English")

    c1, c2, c3, c4 = st.columns(4)
    with c1:
        st.markdown(f'<div class="metric-card">'
                    f'<div class="metric-value">{flag}</div>'
                    f'<div class="metric-label">{lang_name.split()[0]}</div></div>',
                    unsafe_allow_html=True)
    with c2:
        st.markdown(f'<div class="metric-card">'
                    f'<div class="metric-value">{len(result["retrieved_docs"])}</div>'
                    f'<div class="metric-label">Docs Retrieved</div></div>',
                    unsafe_allow_html=True)
    with c3:
        total_t = sum(result["timings"].values())
        st.markdown(f'<div class="metric-card">'
                    f'<div class="metric-value">{total_t:.2f}s</div>'
                    f'<div class="metric-label">Total Time</div></div>',
                    unsafe_allow_html=True)
    with c4:
        top_score = (max(r["score"] for r in result["retrieved_docs"])
                     if result["retrieved_docs"] else 0)
        st.markdown(f'<div class="metric-card">'
                    f'<div class="metric-value">{top_score:.3f}</div>'
                    f'<div class="metric-label">Top Score</div></div>',
                    unsafe_allow_html=True)

    st.markdown("---")
    left_col, right_col = st.columns(2)

    with left_col:
        st.markdown("#### 🔍 Language Detection")
        ld = result["lang_details"]
        st.markdown(f"""
        <div class="lang-detect-box">
            <b>Method:</b> {ld.get('method', 'N/A')}<br>
            <b>Final Language:</b> {flag} {lang_name}<br>
            <b>Unicode Result:</b> {ld.get('unicode_result', 'N/A')}<br>
            <b>Langdetect Result:</b> {ld.get('fallback_result', 'N/A')}
        </div>""", unsafe_allow_html=True)

        st.markdown("**📊 Unicode Analysis**")
        st.markdown(f"""
        <div class="unicode-box">
            Total chars   : {ld.get('total_chars', 0)}<br>
            Sinhala chars : {ld.get('sinhala_chars', 0)}
                ({ld.get('sinhala_ratio', 0):.1%}) [U+0D80–U+0DFF]<br>
            Tamil chars   : {ld.get('tamil_chars', 0)}
                ({ld.get('tamil_ratio', 0):.1%}) [U+0B80–U+0BFF]<br>
            Threshold     : 10%
        </div>""", unsafe_allow_html=True)

        st.markdown("---")
        st.markdown("#### 🔄 Translation (via Ollama LLM)")
        if result["lang_code"] == "en":
            st.success("No translation needed (English query)")
        else:
            st.markdown(f"""
            <div class="lang-detect-box">
                <b>Original:</b> {result['query']}<br><br>
                <b>→ English:</b> {result['english_query']}
            </div>""", unsafe_allow_html=True)

        st.markdown("---")
        st.markdown("#### ⏱️ Timing Breakdown")
        timing_labels = {
            "lang_detection"       : "Language Detection",
            "translation"          : "Query Translation (LLM)",
            "retrieval"            : "FAISS Retrieval",
            "llm"                  : "LLM Generation",
            "response_translation" : "Response Translation (LLM)"
        }
        for key, label in timing_labels.items():
            t = result["timings"].get(key, 0)
            if t > 0:
                st.markdown(
                    f'<div class="pipeline-step">'
                    f'<span>⏱️</span><span style="flex:1">{label}</span>'
                    f'<span style="color:#68d391; font-weight:700;">{t:.3f}s</span>'
                    f'</div>', unsafe_allow_html=True)

    with right_col:
        st.markdown("#### 📚 Retrieved Documents (FAISS)")
        if not result["retrieved_docs"]:
            st.warning("No documents retrieved above threshold.")
        else:
            for r in result["retrieved_docs"]:
                doc   = r["document"]
                score = r["score"]
                color = "#68d391" if score > 0.6 else \
                        "#f6ad55" if score > 0.4 else "#fc8181"
                cat_emoji = {
                    "health": "🏥", "education": "🎓",
                    "government": "🏛️", "agriculture": "🌾", "finance": "💰"
                }.get(doc["category"], "📄")
                st.markdown(f"""
                <div class="retrieved-doc">
                    <div style="display:flex; justify-content:space-between;
                                align-items:center; margin-bottom:6px;">
                        <span class="doc-title">{cat_emoji} {doc['title']}</span>
                        <span class="doc-score" style="color:{color};">{score:.4f}</span>
                    </div>
                    <span class="doc-category">{doc['category'].upper()}</span>
                    <span style="color:#718096; font-size:0.75rem; margin-left:8px;">
                        FAISS ID: {r['faiss_id']}
                    </span>
                    <div class="doc-content">
                        {doc['content'][:200]}{'...' if len(doc['content']) > 200 else ''}
                    </div>
                </div>""", unsafe_allow_html=True)

        st.markdown("---")
        st.markdown("#### 🤖 LLM Response (English)")
        if result.get("english_response"):
            st.markdown(f'<div class="lang-detect-box">{result["english_response"]}</div>',
                        unsafe_allow_html=True)

        if result["lang_code"] != "en":
            st.markdown(f"#### {flag} Final Response ({lang_name.split()[0]})")
            st.markdown(
                f'<div class="lang-detect-box" style="border-color:#68d391;">'
                f'{result["final_response"]}</div>',
                unsafe_allow_html=True
            )


# ============================================================
# KNOWLEDGE BASE TAB
# ============================================================

def render_knowledge_tab(model):
    st.markdown('<div class="section-header">🗄️ Knowledge Base Management</div>',
                unsafe_allow_html=True)

    tab_view, tab_add = st.tabs(["📋 View Documents", "➕ Add Document"])

    with tab_view:
        rows = get_all_documents_display()
        total_docs, _ = get_db_stats()

        all_cats    = sorted(set(r[1] for r in rows))
        filter_cats = st.multiselect("Filter by Category", options=all_cats,
                                     default=all_cats, key="cat_filter")
        filtered = [r for r in rows if r[1] in filter_cats]
        st.caption(f"Showing {len(filtered)} of {total_docs} documents")

        for row in filtered:
            doc_id, category, title, content = row
            cat_emoji = {
                "health": "🏥", "education": "🎓",
                "government": "🏛️", "agriculture": "🌾", "finance": "💰"
            }.get(category, "📄")

            with st.expander(f"{cat_emoji} [{doc_id}] {title}"):
                st.markdown(f"**Category:** `{category.upper()}`")
                st.markdown(f"**Content:** {content}")

                if st.button(f"🗑️ Delete Document #{doc_id}",
                             key=f"del_{doc_id}", type="secondary"):
                    delete_document_from_db(doc_id)
                    with st.spinner("Rebuilding FAISS index after deletion..."):
                        new_idx, new_ds = rebuild_index(model)
                        st.session_state.faiss_index = new_idx
                        st.session_state.doc_store   = new_ds
                        initialize_faiss.clear()
                    st.success(f"✅ Document #{doc_id} deleted & index rebuilt!")
                    st.rerun()

    with tab_add:
        st.markdown("#### ➕ Add New English Document")
        st.info("📌 All documents must be in **English**. "
                "The bot translates queries to English before searching.")

        with st.form("add_doc_form", clear_on_submit=True):
            new_category = st.selectbox(
                "Category",
                options=["health", "education", "government",
                         "agriculture", "finance", "other"]
            )
            new_title   = st.text_input("Title", placeholder="e.g., Typhoid Fever Symptoms")
            new_content = st.text_area("Content (English)",
                                       placeholder="Enter document content in English...",
                                       height=150)
            submitted = st.form_submit_button("➕ Add Document", type="primary")

            if submitted:
                if not new_title.strip() or not new_content.strip():
                    st.error("❌ Title and Content are required.")
                else:
                    new_id  = add_document_to_db(
                        new_category.strip(), new_title.strip(), new_content.strip()
                    )
                    new_doc = {
                        "db_id"    : new_id,
                        "category" : new_category,
                        "title"    : new_title.strip(),
                        "content"  : new_content.strip(),
                        "full_text": f"passage: {new_title.strip()}. {new_content.strip()}"
                    }
                    new_emb = model.encode(
                        [new_doc["full_text"]],
                        normalize_embeddings=True,
                        convert_to_numpy=True
                    ).astype(np.float32)

                    st.session_state.faiss_index.add(new_emb)
                    st.session_state.doc_store.append(new_doc)
                    save_faiss_index(st.session_state.faiss_index, st.session_state.doc_store)

                    st.success(
                        f"✅ Document added! "
                        f"FAISS now has {st.session_state.faiss_index.ntotal} vectors."
                    )
                    st.rerun()


# ============================================================
# ABOUT TAB
# ============================================================

def render_about_tab():
    st.markdown('<div class="section-header">ℹ️ About This System</div>',
                unsafe_allow_html=True)

    c1, c2 = st.columns(2)

    with c1:
        st.markdown("""
        #### 🏗️ Architecture
        ```
        User Query (EN / TA / SI)
               ↓
        Unicode Language Detection
          [Sinhala: U+0D80–U+0DFF]
          [Tamil  : U+0B80–U+0BFF]
               ↓
        Ollama LLM → Translate to English
          (fully offline, no Google Translate)
               ↓
        multilingual-e5-small
          (384-dim embedding)
               ↓
        FAISS IndexFlatIP
          (Cosine Similarity)
               ↓
        Top-K Documents Retrieved
               ↓
        Ollama LLM — Grounded Answer (EN)
               ↓
        Ollama LLM → Translate to User Lang
               ↓
        Final Response
        ```
        """)

    with c2:
        st.markdown("""
        #### 🛠️ Tech Stack

        | Component | Technology |
        |-----------|-----------|
        | Vector DB | FAISS IndexFlatIP |
        | Metadata DB | SQLite |
        | Embeddings | multilingual-e5-small |
        | Lang Detection | Unicode + langdetect |
        | Translation | Ollama LLM (offline) |
        | LLM | Ollama REST API |
        | UI | Streamlit |

        #### 🌐 Language Support

        | Language | Script | Unicode Range |
        |----------|--------|--------------|
        | English | Latin | U+0041–U+007A |
        | Tamil | Tamil | U+0B80–U+0BFF |
        | Sinhala | Sinhala | U+0D80–U+0DFF |

        #### 🔑 Key Features
        - ✅ Fully offline (zero cloud APIs)
        - ✅ LLM-powered translation (no Google Translate)
        - ✅ Unicode-based Sinhala/Tamil detection
        - ✅ FAISS exact vector search
        - ✅ Incremental document addition
        - ✅ Real-time pipeline inspection
        """)

    st.markdown("---")
    st.markdown("""
    #### ⚠️ Important Notes

    1. **All translation is done by Ollama LLM** → no internet required
    2. **DB content is English only** → queries are translated before searching
    3. **Unicode detection is priority** → langdetect only used as English fallback
    4. **FAISS IndexFlatIP** → exact search, supports incremental add
    5. **Deletion requires rebuild** → FAISS does not support vector deletion
    6. **Ollama status uses REST API** → `GET /api/tags` on localhost:11434
    7. **Run `ollama serve` before starting** if Ollama shows offline
    """)


# ============================================================
# MAIN
# ============================================================

def main():
    init_session_state()

    st.markdown("""
    <div class="header-banner">
        <div class="header-title">🌐 TRILINGUAL RAG BOT</div>
        <div class="header-subtitle">
            English · Tamil · Sinhala &nbsp;|&nbsp;
            FAISS + SQLite + Ollama &nbsp;|&nbsp; 🔒 Fully Offline
        </div>
        <div class="lang-badges">
            <span class="lang-badge">🇬🇧 English</span>
            <span class="lang-badge">🇮🇳 தமிழ் Tamil</span>
            <span class="lang-badge">🇱🇰 සිංහල Sinhala</span>
            <span class="lang-badge">🔒 No Cloud APIs</span>
            <span class="lang-badge">⚡ FAISS Vector Search</span>
        </div>
    </div>
    """, unsafe_allow_html=True)

    with st.spinner("⏳ Loading multilingual embedding model..."):
        model = load_embedding_model()

    if st.session_state.faiss_index is None:
        with st.spinner("⏳ Initializing FAISS index..."):
            idx, ds = initialize_faiss(model)
            st.session_state.faiss_index = idx
            st.session_state.doc_store   = ds

    selected_model, top_k, score_threshold = render_sidebar(model)

    tab_chat, tab_inspect, tab_kb, tab_about = st.tabs([
        "💬 Chat",
        "🔬 Pipeline Inspector",
        "🗄️ Knowledge Base",
        "ℹ️ About"
    ])

    with tab_chat:
        render_chat_tab(
            model, st.session_state.faiss_index,
            st.session_state.doc_store,
            top_k, score_threshold, selected_model
        )
    with tab_inspect:
        render_inspector_tab()
    with tab_kb:
        render_knowledge_tab(model)
    with tab_about:
        render_about_tab()


if __name__ == "__main__":
    main()

Writing app.py


In [ ]:
# Run this in a new notebook cell
!streamlit run app.py